In [ ]:
# RUN-MODE UTILITIES (safe)
import os
EQUIPAY_MODE = globals().get('EQUIPAY_MODE', os.environ.get('EQUIPAY_MODE','FAST'))

# Better defaults: 10K for FAST mode, 100K for FULL mode  
if EQUIPAY_MODE == 'FAST':
    N_SAMPLES = globals().get('N_SAMPLES', 10000)
else:
    N_SAMPLES = globals().get('N_SAMPLES', 100000)

N_BOOTSTRAP = globals().get('N_BOOTSTRAP', 100)
MAX_ITER = globals().get('MAX_ITER', 200)
N_ESTIMATORS = globals().get('N_ESTIMATORS', 50)

def enforce_fast_sample(df, n=None, seed=42):
    if EQUIPAY_MODE == 'FAST' and n is not None:
        return df.sample(n=min(len(df), n), random_state=seed)
    return df

# Conservative default: in FAST mode we skip small groups to avoid heavy per-group loops.
# In FULL mode we allow small groups to be processed for comprehensive analysis.
MIN_GROUP_N = 1000 if EQUIPAY_MODE == 'FAST' else 30
print(f"EQUIPAY_MODE={EQUIPAY_MODE}; MIN_GROUP_N={MIN_GROUP_N}")

from src.notebook_utils import ensure_store_and_sample, get_sample_from_store, safe_weight_col
store, df_sample = ensure_store_and_sample(sample_limit=N_SAMPLES)
weight_col = safe_weight_col(df_sample)

# Data loading feedback
print(f"✓ Loaded {len(df_sample):,} records")
print(f"✓ Dataset has {len(df_sample.columns)} columns")
if 'SURVYEAR' in df_sample.columns:
    year_range = f"{df_sample['SURVYEAR'].min():.0f}-{df_sample['SURVYEAR'].max():.0f}"
    print(f"✓ Year range: {year_range}")

# Notebook 04: Fairness Evaluation

**EquiPay Canada**

## Survey-Weighted Fairness Analysis

This notebook evaluates model fairness using **survey weights (FINALWT)** to ensure all metrics represent the Canadian labor force population, not just the sample.

**Key Weighted Metrics:**
- Weighted Disparate Impact Ratio
- Weighted RMSE by gender group
- Weighted bias amplification check

In [ ]:
# ============================================================================
# SETUP: Import Libraries with Weighted ML Utilities
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.constants import COLS, normalize_column_names

# Import weighted ML utilities
from src.ml_utils import WeightedMLSplitter, WeightedMetrics, WeightedGapAnalysis

# NEW: Import gap analysis for intersectional fairness
from src.gap_analysis import IntersectionalAnalyzer
from src.gap_analysis.core import weighted_mean

print('✓ Setup complete')
print('✓ Using survey weights (FINALWT) for population-level fairness metrics')
print('✓ IntersectionalAnalyzer loaded for compound discrimination detection')

In [ ]:
# ============================================================================
# DATA LOADING VIA EquiPayDataStore (6-Layer Architecture)
# ============================================================================

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_store import EquiPayDataStore, Agg, Func

# Initialize data store with new architecture
store = EquiPayDataStore(
    parquet_path=str(PROJECT_ROOT / "data" / "parquet"),
    memory_limit_mb=1000,
    enable_cache=True
)

# Load data with valid wages using new API
try:
    df = store.query().where('HRLYEARN > 0').to_pandas()
except Exception:
    df = get_sample_from_store(query='HRLYEARN > 0', limit=N_SAMPLES)

df = normalize_column_names(df)

# Identify columns
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'
wage_col = 'REAL_HRLYEARN' if 'REAL_HRLYEARN' in df.columns else 'HRLYEARN'
weight_col = COLS.FINAL_WEIGHT if COLS.FINAL_WEIGHT in df.columns else 'FINALWT'

# VALIDATE survey weights are present (MANDATORY)
if weight_col not in df.columns:
    raise ValueError(f"Survey weight column '{weight_col}' not found! MANDATORY for population inference.")

df['IS_FEMALE'] = (df[gender_col] == 2).astype(int)

# Add immigration status for intersectional analysis
if 'IMMIG' in df.columns:
    df['IS_IMMIGRANT'] = (df['IMMIG'].isin([1, 2, 3])).astype(int)

# Immigration status (IMMIG: 1-3 = immigrant, 4 = Canadian-born)
if 'IMMIG' in df.columns:
    df['IS_IMMIGRANT'] = (df['IMMIG'].isin([1, 2, 3])).astype(int)
    df['RECENT_IMMIGRANT'] = (df['IMMIG'] == 1).astype(int)  # <5 years
    print(f"✓ IMMIG available: {df['IS_IMMIGRANT'].mean()*100:.1f}% immigrants")
else:
    df['IS_IMMIGRANT'] = 0
    df['RECENT_IMMIGRANT'] = 0

# CMA (Census Metropolitan Area: 0 = non-CMA, 1-9 = CMAs)
if 'CMA' in df.columns:
    df['IS_URBAN'] = (df['CMA'] > 0).astype(int)
    print(f"✓ CMA available: {df['IS_URBAN'].mean()*100:.1f}% urban workers")
else:
    df['IS_URBAN'] = 0

# Firm size (FIRMSIZE)
if 'FIRMSIZE' in df.columns:
    df['LARGE_FIRM'] = (df['FIRMSIZE'] >= 3).astype(int)  # 100+ employees
    print(f"✓ FIRMSIZE available: {df['LARGE_FIRM'].mean()*100:.1f}% in large firms")
else:
    df['LARGE_FIRM'] = 0

# Family status (AGYOWNK - age of youngest child)
if 'AGYOWNK' in df.columns:
    df['HAS_YOUNG_CHILD'] = (df['AGYOWNK'].isin([1, 2, 3])).astype(int)  # Ages 0-5
    print(f"✓ AGYOWNK available: {df['HAS_YOUNG_CHILD'].mean()*100:.1f}% with young children")
else:
    df['HAS_YOUNG_CHILD'] = 0

# Public sector (COWMAIN: 3-4 = public sector)
if 'COWMAIN' in df.columns:
    df['PUBLIC_SECTOR'] = (df['COWMAIN'].isin([3, 4])).astype(int)
    print(f"✓ COWMAIN available: {df['PUBLIC_SECTOR'].mean()*100:.1f}% public sector")
else:
    df['PUBLIC_SECTOR'] = 0

print(f'\n✓ Loaded {len(df):,} records via DuckDB (6-Layer Architecture)')
print(f'✓ Survey weight column: {weight_col}')
print(f'  Total population weight: {df[weight_col].sum():,.0f}')
print(f'✓ Using REAL hourly earnings (inflation-adjusted)')

In [ ]:
# ============================================================================
# WEIGHTED TRAIN/TEST SPLIT
# ============================================================================

features = [c for c in [COLS.EDUC, COLS.AGE_12, COLS.NOC_10, COLS.PROV, 'IS_FEMALE'] if c in df.columns or c == 'IS_FEMALE']
df_clean = df[features + [wage_col, weight_col]].dropna()

# Use log wages for better model performance
df_clean['LOG_WAGE'] = np.log(df_clean[wage_col].clip(lower=1))

print(f"Clean dataset: {len(df_clean):,} records")

# Use WeightedMLSplitter for proper stratified split
splitter = WeightedMLSplitter(
    data=df_clean,
    weight_col=weight_col,
    stratify_cols=['IS_FEMALE']  # Stratify by gender for fairness
)
splits = splitter.create_splits(test_size=0.2, val_size=0.0)  # No validation for this notebook

# Extract splits
X_train = splits['X_train'][features]
X_test = splits['X_test'][features]
y_train = splits['y_train']
y_train = splits['y_train']
y_test = splits['y_test']
w_train = splits['w_train']
w_test = splits['w_test']

# Also need log wages
y_train_log = np.log(y_train.clip(lower=1))
y_test_log = np.log(y_test.clip(lower=1))

# Sensitive attribute (gender)
s_train = X_train['IS_FEMALE'].values
s_test = X_test['IS_FEMALE'].values

print(f'✓ Weighted Train: {len(X_train):,} (pop weight: {w_train.sum():,.0f})')
print(f'✓ Weighted Test: {len(X_test):,} (pop weight: {w_test.sum():,.0f})')

In [ ]:
# ============================================================================
# MODEL TRAINING WITH SURVEY WEIGHTS
# ============================================================================

# Remove IS_FEMALE from features for prediction (it's our protected attribute)
feature_cols = [c for c in features if c != 'IS_FEMALE']
X_train_pred = X_train[feature_cols]
X_test_pred = X_test[feature_cols]

# Train with sample weights
model = GradientBoostingRegressor(n_estimators=N_ESTIMATORS, max_depth=4, random_state=42)
model.fit(X_train_pred, y_train_log, sample_weight=w_train)

y_pred_log = model.predict(X_test_pred)
y_pred = np.exp(y_pred_log)  # Back to dollar scale

# Evaluate with WEIGHTED metrics
metrics = WeightedMetrics.evaluate(y_test_log, y_pred_log, w_test)

print("=" * 60)
print("MODEL PERFORMANCE (WEIGHTED)")
print("=" * 60)
print(f'Weighted R²: {metrics["weighted_r2"]:.4f}')
print(f'Weighted RMSE (log): {metrics["weighted_rmse"]:.4f}')
print(f'Unweighted R²: {metrics["unweighted_r2"]:.4f}')

In [ ]:
# ============================================================================
# WEIGHTED FAIRNESS METRICS
# ============================================================================

male_mask = s_test == 0
female_mask = s_test == 1

# Weighted means for predictions and actuals
male_pred_weighted = np.average(y_pred[male_mask], weights=w_test[male_mask])
female_pred_weighted = np.average(y_pred[female_mask], weights=w_test[female_mask])

male_actual_weighted = np.average(y_test.values[male_mask], weights=w_test[male_mask])
female_actual_weighted = np.average(y_test.values[female_mask], weights=w_test[female_mask])

# Weighted Disparate Impact Ratio
di_ratio_weighted = female_pred_weighted / male_pred_weighted

# Weighted RMSE by group
rmse_m_weighted = WeightedMetrics.weighted_rmse(
    y_test.values[male_mask], y_pred[male_mask], w_test[male_mask]
)
rmse_f_weighted = WeightedMetrics.weighted_rmse(
    y_test.values[female_mask], y_pred[female_mask], w_test[female_mask]
)

print("=" * 70)
print("WEIGHTED FAIRNESS RESULTS (Population-Level)")
print("=" * 70)
print(f'\nPredicted Wages (weighted means):')
print(f'  Male: ${male_pred_weighted:.2f}/hr')
print(f'  Female: ${female_pred_weighted:.2f}/hr')
print(f'\nActual Wages (weighted means):')
print(f'  Male: ${male_actual_weighted:.2f}/hr')
print(f'  Female: ${female_actual_weighted:.2f}/hr')
print(f'\nWeighted Disparate Impact Ratio: {di_ratio_weighted:.3f}')
print(f'4/5ths Rule (≥0.8): {"✅ PASS" if di_ratio_weighted >= 0.8 else "❌ FAIL"}')
print(f'\nWeighted RMSE by Group:')
print(f'  Male: ${rmse_m_weighted:.2f}')
print(f'  Female: ${rmse_f_weighted:.2f}')
print(f'  Error Parity Ratio: {rmse_f_weighted/rmse_m_weighted:.3f}')

In [ ]:
# ============================================================================
# VISUALIZATION: Weighted Prediction Distributions
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with weighted counts
ax = axes[0]
bins = np.linspace(y_pred.min(), np.percentile(y_pred, 99), 40)

# Use histogram weights for proper population representation
ax.hist(y_pred[male_mask], bins=bins, weights=w_test[male_mask]/1000, 
        alpha=0.6, label='Male', color='steelblue')
ax.hist(y_pred[female_mask], bins=bins, weights=w_test[female_mask]/1000, 
        alpha=0.6, label='Female', color='coral')
ax.axvline(male_pred_weighted, color='steelblue', ls='--', lw=2, label=f'Male mean: ${male_pred_weighted:.2f}')
ax.axvline(female_pred_weighted, color='coral', ls='--', lw=2, label=f'Female mean: ${female_pred_weighted:.2f}')
ax.set_xlabel('Predicted Wage ($/hr)')
ax.set_ylabel('Population Weight (thousands)')
ax.set_title('Weighted Prediction Distribution by Gender', fontweight='bold')
ax.legend()

# Fairness metrics bar chart
ax = axes[1]
metrics_names = ['Disparate\nImpact', 'Error\nParity', '4/5ths\nThreshold']
metrics_values = [di_ratio_weighted, rmse_f_weighted/rmse_m_weighted, 0.8]
colors = ['#2ecc71' if v >= 0.8 else '#e74c3c' for v in metrics_values[:2]] + ['#95a5a6']

bars = ax.bar(metrics_names, metrics_values, color=colors)
ax.axhline(y=0.8, color='black', ls='--', lw=1.5, label='4/5ths threshold')
ax.axhline(y=1.0, color='gray', ls=':', lw=1, label='Parity (1.0)')
ax.set_ylabel('Ratio')
ax.set_title('Weighted Fairness Metrics', fontweight='bold')
ax.legend()
for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/fairness_weighted_metrics.png', dpi=150, 
            bbox_inches='tight', facecolor='white')
plt.show()

print("📊 Figure saved: reports/figures/fairness_weighted_metrics.png")

In [ ]:
# ============================================================================
# BIAS AMPLIFICATION CHECK
# ============================================================================

print("=" * 70)
print("BIAS AMPLIFICATION CHECK (Does Model Amplify Wage Gap?)")
print("=" * 70)

# Use WeightedGapAnalysis for proper bias detection
analyzer = WeightedGapAnalysis(
    y_true=y_test.values,
    y_pred=y_pred,
    weights=w_test,
    group_indicator=s_test + 1,  # 1=Male, 2=Female (to match SEX coding)
    group_names={1: 'Male', 2: 'Female'}
)

actual_gap = analyzer.compute_gap(use_predictions=False)
predicted_gap = analyzer.compute_gap(use_predictions=True)
bias_check = analyzer.check_bias_amplification()

print(f'\nActual wage gap (weighted): {(1 - np.exp(actual_gap)) * 100:.1f}%')
print(f'Predicted wage gap (weighted): {(1 - np.exp(predicted_gap)) * 100:.1f}%')
print(f'\nBias Amplification: {"⚠️ YES - Model amplifies existing bias" if bias_check["amplifies_bias"] else "✅ NO - Model does not amplify bias"}')

# Summary
print("\n" + "=" * 70)
print("WEIGHTED FAIRNESS SUMMARY")
print("=" * 70)
print(f"✓ All metrics computed using survey weights (FINALWT)")
print(f"✓ Results represent Canadian labor force population")
print(f"\nKey Results:")
print(f"  Weighted Disparate Impact Ratio: {di_ratio_weighted:.3f}")
print(f"  4/5ths Rule Compliance: {'COMPLIANT ✅' if di_ratio_weighted >= 0.8 else 'NON-COMPLIANT ❌'}")
print(f"  Weighted Error Parity: {rmse_f_weighted/rmse_m_weighted:.3f}")
print(f"  Bias Amplification: {'YES ⚠️' if bias_check['amplifies_bias'] else 'NO ✅'}")
print("=" * 70)

## 4. NEW: Intersectional Fairness Analysis

This section applies **intersectional analysis** to detect **compound discrimination** (Crenshaw, 1989). Single-axis fairness metrics may hide disparities affecting subgroups defined by multiple protected attributes (e.g., immigrant women).

In [ ]:
# ============================================================================
# INTERSECTIONAL FAIRNESS: Gender × Immigration Status
# ============================================================================

print("=" * 70)
print("INTERSECTIONAL FAIRNESS ANALYSIS")
print("=" * 70)

# Define intersectional groups
df['INTERSECT_GROUP'] = 'Native-Born Male'
df.loc[(df['IS_FEMALE'] == 0) & (df['IS_IMMIGRANT'] == 1), 'INTERSECT_GROUP'] = 'Immigrant Male'
df.loc[(df['IS_FEMALE'] == 1) & (df['IS_IMMIGRANT'] == 0), 'INTERSECT_GROUP'] = 'Native-Born Female'
df.loc[(df['IS_FEMALE'] == 1) & (df['IS_IMMIGRANT'] == 1), 'INTERSECT_GROUP'] = 'Immigrant Female'

# Compute weighted statistics by intersectional group
intersect_stats = []
for group in ['Native-Born Male', 'Immigrant Male', 'Native-Born Female', 'Immigrant Female']:
    mask = df['INTERSECT_GROUP'] == group
    subset = df[mask]
    if len(subset) > 0:
        weighted_mean_wage = np.average(subset[wage_col], weights=subset[weight_col])
        weighted_median = subset[wage_col].quantile(0.5)  # Approximate
        pop_weight = subset[weight_col].sum()
        intersect_stats.append({
            'Group': group,
            'N (unweighted)': len(subset),
            'Population Weight': pop_weight,
            'Weighted Mean Wage': weighted_mean_wage,
        })

intersect_df = pd.DataFrame(intersect_stats)

# Reference group: Native-Born Male
ref_wage = intersect_df.loc[intersect_df['Group'] == 'Native-Born Male', 'Weighted Mean Wage'].values[0]
intersect_df['Gap vs Reference'] = (1 - intersect_df['Weighted Mean Wage'] / ref_wage) * 100

print("\nIntersectional Wage Gaps (vs Native-Born Male reference):\n")
print(intersect_df.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

# Test for intersectional penalty (compound discrimination)
# Immigrant Women should face penalty > (Gender gap) + (Immigrant gap)
gender_gap = intersect_df.loc[intersect_df['Group'] == 'Native-Born Female', 'Gap vs Reference'].values[0]
immig_gap = intersect_df.loc[intersect_df['Group'] == 'Immigrant Male', 'Gap vs Reference'].values[0]
imm_women_gap = intersect_df.loc[intersect_df['Group'] == 'Immigrant Female', 'Gap vs Reference'].values[0]

additive_penalty = gender_gap + immig_gap
intersectional_penalty = imm_women_gap - additive_penalty

print(f"\n{'='*70}")
print("COMPOUND DISCRIMINATION TEST")
print("=" * 70)
print(f"Gender gap (Native-Born Women): {gender_gap:.1f}%")
print(f"Immigration gap (Immigrant Men): {immig_gap:.1f}%")
print(f"Expected additive penalty: {additive_penalty:.1f}%")
print(f"Observed Immigrant Women gap: {imm_women_gap:.1f}%")
print(f"\n→ Intersectional penalty: {intersectional_penalty:+.1f}%")
if intersectional_penalty > 1:
    print("⚠️  COMPOUND DISCRIMINATION DETECTED: Immigrant women face penalty beyond sum of individual effects")
else:
    print("✅ No significant compound discrimination - effects are approximately additive")

In [ ]:
# ============================================================================
# INTERSECTIONAL ANALYZER (Using gap_analysis module)
# ============================================================================

# Use the IntersectionalAnalyzer for comprehensive analysis
try:
    analyzer = IntersectionalAnalyzer(df, weight_col=weight_col)
    
    # Define protected attributes for analysis
    protected_attrs = ['IS_FEMALE', 'IS_IMMIGRANT']
    
    # Add education if available
    if 'EDUC' in df.columns:
        df['LOW_EDUCATION'] = (df['EDUC'] <= 2).astype(int)  # High school or less
        protected_attrs.append('LOW_EDUCATION')
    
    # Run full intersectional analysis
    results = analyzer.analyze(
        outcome_col=wage_col,
        protected_attrs=protected_attrs,
        reference_groups={'IS_FEMALE': 0, 'IS_IMMIGRANT': 0, 'LOW_EDUCATION': 0}
    )
    
    print("\n" + "=" * 70)
    print("FULL INTERSECTIONAL ANALYSIS RESULTS")
    print("=" * 70)
    
    if 'intersection_gaps' in results:
        print("\nIntersection-specific gaps:")
        for intersection, gap_info in results['intersection_gaps'].items():
            print(f"  {intersection}: {gap_info['gap_pct']:.1f}% gap, N={gap_info['n']:,}")
    
    if 'compound_discrimination' in results:
        print("\nCompound Discrimination Tests:")
        for test_name, test_result in results['compound_discrimination'].items():
            status = "⚠️ DETECTED" if test_result['significant'] else "✅ Not significant"
            print(f"  {test_name}: {status} (penalty={test_result['penalty']:.1f}%)")
            
except Exception as e:
    print(f"Note: IntersectionalAnalyzer not fully initialized: {e}")
    print("Using manual calculations above.")

In [ ]:
# ============================================================================
# INTERSECTIONAL FAIRNESS VISUALIZATION
# ============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Intersectional wage gaps
ax1 = axes[0]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
bars = ax1.barh(intersect_df['Group'], intersect_df['Gap vs Reference'], color=colors)
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.set_xlabel('Wage Gap vs Native-Born Male (%)')
ax1.set_title('Intersectional Wage Gaps\n(Gender × Immigration Status)')
ax1.set_xlim(-5, max(intersect_df['Gap vs Reference']) * 1.2)

# Add value labels
for bar, gap in zip(bars, intersect_df['Gap vs Reference']):
    ax1.text(gap + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{gap:.1f}%', va='center', fontsize=10)

# Plot 2: Additive vs Observed penalty
ax2 = axes[1]
categories = ['Gender Effect\n(Native Women)', 'Immigration Effect\n(Immigrant Men)', 
              'Expected Additive', 'Observed\n(Immigrant Women)']
values = [gender_gap, immig_gap, additive_penalty, imm_women_gap]
colors2 = ['#3498db', '#e67e22', '#95a5a6', '#9b59b6']
bars2 = ax2.bar(categories, values, color=colors2, edgecolor='black', linewidth=0.5)
ax2.axhline(y=additive_penalty, color='gray', linestyle='--', alpha=0.7, label='Expected (additive)')
ax2.set_ylabel('Wage Gap (%)')
ax2.set_title('Compound Discrimination Test')

# Annotate the intersectional penalty
if intersectional_penalty > 0:
    ax2.annotate(f'Intersectional\nPenalty: +{intersectional_penalty:.1f}%',
                 xy=(3, imm_women_gap), xytext=(3.5, imm_women_gap * 0.7),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=9, color='red')

plt.tight_layout()
plt.savefig('../reports/figures/intersectional_fairness.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved: reports/figures/intersectional_fairness.png")

## 5. NEW: CMA × Gender and Firm Size × Gender Fairness Analysis

Using the new LFS columns to analyze intersectional fairness across:
- **CMA × Gender**: Urban vs Rural wage gap differences
- **Firm Size × Gender**: Large vs small employer discrimination patterns
- **Immigration × Gender × CMA**: Triple intersection analysis

In [ ]:
# ============================================================================
# CMA × GENDER FAIRNESS ANALYSIS
# ============================================================================

print("=" * 70)
print("CMA × GENDER INTERSECTIONAL FAIRNESS")
print("=" * 70)

# CMA code meanings (PUMF simplified):
CMA_NAMES = {
    0: 'Non-CMA (Rural)',
    1: 'Toronto CMA',
    2: 'Montréal CMA', 
    3: 'Vancouver CMA',
    4: 'Ottawa-Gatineau CMA',
    5: 'Calgary CMA',
    6: 'Edmonton CMA',
    7: 'Québec City CMA',
    8: 'Winnipeg CMA',
    9: 'Other CMAs'
}

if 'CMA' in df.columns:
    cma_gender_stats = []
    
    for cma_code in sorted(df['CMA'].dropna().unique()):
        for gender in [0, 1]:  # 0=Male, 1=Female
            subset = df[(df['CMA'] == cma_code) & (df['IS_FEMALE'] == gender)]
            if len(subset) >= 50:
                weighted_wage = np.average(subset[wage_col], weights=subset[weight_col])
                pop_weight = subset[weight_col].sum()
                cma_gender_stats.append({
                    'CMA': CMA_NAMES.get(int(cma_code), f'CMA {int(cma_code)}'),
                    'CMA_Code': int(cma_code),
                    'Gender': 'Female' if gender == 1 else 'Male',
                    'Weighted_Wage': weighted_wage,
                    'Pop_Weight': pop_weight,
                    'N': len(subset)
                })
    
    cma_df = pd.DataFrame(cma_gender_stats)
    
    # Calculate gap by CMA
    cma_gaps = []
    for cma_code in cma_df['CMA_Code'].unique():
        cma_data = cma_df[cma_df['CMA_Code'] == cma_code]
        male_wage = cma_data[cma_data['Gender'] == 'Male']['Weighted_Wage'].values
        female_wage = cma_data[cma_data['Gender'] == 'Female']['Weighted_Wage'].values
        
        if len(male_wage) > 0 and len(female_wage) > 0:
            gap_pct = (male_wage[0] - female_wage[0]) / male_wage[0] * 100
            cma_gaps.append({
                'CMA': CMA_NAMES.get(cma_code, f'CMA {cma_code}'),
                'Male_Wage': male_wage[0],
                'Female_Wage': female_wage[0],
                'Gap_%': gap_pct,
                'DI_Ratio': female_wage[0] / male_wage[0]
            })
    
    cma_gap_df = pd.DataFrame(cma_gaps).sort_values('Gap_%', ascending=False)
    
    print("\nGender Wage Gap by Census Metropolitan Area:")
    print("-" * 70)
    print(f"{'CMA':<25} {'Male':>10} {'Female':>10} {'Gap %':>10} {'DI Ratio':>10}")
    print("-" * 70)
    for _, row in cma_gap_df.iterrows():
        di_status = "✓" if row['DI_Ratio'] >= 0.8 else "✗"
        print(f"{row['CMA']:<25} ${row['Male_Wage']:>9.2f} ${row['Female_Wage']:>9.2f} {row['Gap_%']:>9.1f}% {row['DI_Ratio']:>9.3f} {di_status}")
    
    print(f"\n4/5ths Rule (DI ≥ 0.8): {(cma_gap_df['DI_Ratio'] >= 0.8).sum()}/{len(cma_gap_df)} CMAs pass")
else:
    print("⚠ CMA column not available")

In [ ]:
# ============================================================================
# FIRM SIZE × GENDER FAIRNESS ANALYSIS
# ============================================================================

print("=" * 70)
print("FIRM SIZE × GENDER FAIRNESS ANALYSIS")
print("=" * 70)

FIRMSIZE_NAMES = {
    1: 'Less than 20 employees',
    2: '20-99 employees',
    3: '100-499 employees',
    4: '500+ employees'
}

if 'FIRMSIZE' in df.columns:
    firmsize_gaps = []
    
    for size in sorted(df['FIRMSIZE'].dropna().unique()):
        subset = df[df['FIRMSIZE'] == size]
        male = subset[subset['IS_FEMALE'] == 0]
        female = subset[subset['IS_FEMALE'] == 1]
        
        if len(male) >= 50 and len(female) >= 50:
            male_wage = np.average(male[wage_col], weights=male[weight_col])
            female_wage = np.average(female[wage_col], weights=female[weight_col])
            gap_pct = (male_wage - female_wage) / male_wage * 100
            
            firmsize_gaps.append({
                'Firm_Size': FIRMSIZE_NAMES.get(int(size), f'Size {int(size)}'),
                'Size_Code': int(size),
                'Male_Wage': male_wage,
                'Female_Wage': female_wage,
                'Gap_%': gap_pct,
                'DI_Ratio': female_wage / male_wage,
                'N_Male': len(male),
                'N_Female': len(female)
            })
    
    firmsize_df = pd.DataFrame(firmsize_gaps)
    
    print("\nGender Wage Gap by Firm Size:")
    print("-" * 75)
    for _, row in firmsize_df.iterrows():
        di_status = "✓ PASS" if row['DI_Ratio'] >= 0.8 else "✗ FAIL"
        print(f"{row['Firm_Size']:<25}: Gap = {row['Gap_%']:.1f}%, DI = {row['DI_Ratio']:.3f} {di_status}")
    
    # Policy insight
    small_firm_gap = firmsize_df[firmsize_df['Size_Code'] == 1]['Gap_%'].values
    large_firm_gap = firmsize_df[firmsize_df['Size_Code'] >= 3]['Gap_%'].mean()
    
    if len(small_firm_gap) > 0:
        print(f"\n📊 Policy Insight:")
        print(f"   Small firm (<20) gap: {small_firm_gap[0]:.1f}%")
        print(f"   Large firm (100+) gap: {large_firm_gap:.1f}%")
        if small_firm_gap[0] > large_firm_gap:
            print(f"   → Small firms have LARGER gender gap ({small_firm_gap[0] - large_firm_gap:.1f}pp more)")
        else:
            print(f"   → Large firms have LARGER gender gap ({large_firm_gap - small_firm_gap[0]:.1f}pp more)")
else:
    print("⚠ FIRMSIZE column not available")

In [ ]:
# ============================================================================
# TRIPLE INTERSECTION: IMMIGRATION × GENDER × CMA
# ============================================================================

print("=" * 70)
print("TRIPLE INTERSECTION: IMMIGRATION × GENDER × CMA")
print("=" * 70)

if 'IMMIG' in df.columns and 'CMA' in df.columns:
    # Create 8 groups: 2 (gender) × 2 (immigrant) × 2 (urban/rural)
    df['TRIPLE_GROUP'] = (
        df['IS_FEMALE'].astype(str) + '_' + 
        df['IS_IMMIGRANT'].astype(str) + '_' + 
        df['IS_URBAN'].astype(str)
    )
    
    group_names = {
        '0_0_0': 'Native Male Rural',
        '0_0_1': 'Native Male Urban',
        '0_1_0': 'Immigrant Male Rural',
        '0_1_1': 'Immigrant Male Urban',
        '1_0_0': 'Native Female Rural',
        '1_0_1': 'Native Female Urban',
        '1_1_0': 'Immigrant Female Rural',
        '1_1_1': 'Immigrant Female Urban',
    }
    
    triple_stats = []
    for group_code, group_name in group_names.items():
        subset = df[df['TRIPLE_GROUP'] == group_code]
        if len(subset) >= 30:
            weighted_wage = np.average(subset[wage_col], weights=subset[weight_col])
            triple_stats.append({
                'Group': group_name,
                'Code': group_code,
                'Weighted_Wage': weighted_wage,
                'N': len(subset),
                'Pop_Weight': subset[weight_col].sum()
            })
    
    triple_df = pd.DataFrame(triple_stats).sort_values('Weighted_Wage', ascending=False)
    
    # Reference: Native Male Urban (highest earning group typically)
    ref_wage = triple_df[triple_df['Group'] == 'Native Male Urban']['Weighted_Wage'].values
    if len(ref_wage) > 0:
        ref_wage = ref_wage[0]
        triple_df['Gap_vs_Ref_%'] = (1 - triple_df['Weighted_Wage'] / ref_wage) * 100
        
        print("\nTriple Intersection Wage Gaps (vs Native Male Urban):")
        print("-" * 70)
        for _, row in triple_df.iterrows():
            print(f"{row['Group']:<25}: ${row['Weighted_Wage']:.2f}/hr ({row['Gap_vs_Ref_%']:+.1f}% gap)")
        
        # Test for compound discrimination
        # Get individual effects
        native_female_urban = triple_df[triple_df['Group'] == 'Native Female Urban']['Gap_vs_Ref_%'].values
        immigrant_male_urban = triple_df[triple_df['Group'] == 'Immigrant Male Urban']['Gap_vs_Ref_%'].values
        immigrant_female_urban = triple_df[triple_df['Group'] == 'Immigrant Female Urban']['Gap_vs_Ref_%'].values
        
        if len(native_female_urban) > 0 and len(immigrant_male_urban) > 0 and len(immigrant_female_urban) > 0:
            expected_additive = native_female_urban[0] + immigrant_male_urban[0]
            observed = immigrant_female_urban[0]
            compound_penalty = observed - expected_additive
            
            print(f"\n{'='*70}")
            print("COMPOUND DISCRIMINATION TEST (Urban):")
            print(f"  Gender effect (Native Female): {native_female_urban[0]:.1f}%")
            print(f"  Immigration effect (Immigrant Male): {immigrant_male_urban[0]:.1f}%")
            print(f"  Expected additive: {expected_additive:.1f}%")
            print(f"  Observed (Immigrant Female): {observed:.1f}%")
            print(f"  Compound penalty: {compound_penalty:+.1f}%")
            
            if compound_penalty > 2:
                print("\n⚠️  COMPOUND DISCRIMINATION DETECTED in urban areas")
            else:
                print("\n✅ Effects are approximately additive in urban areas")
else:
    print("⚠ IMMIG or CMA not available for triple intersection analysis")